# Watch a wetting front approach the capillary height

Guess whether a wider vertical tube fills faster during the first few millimetres and whether it eventually rises higher. Edit the first cell, then compare the exact viscous trajectory against the early-time square-root law. The assumed tube is slender, the contact angle constant and the reservoir large.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

radius = 100e-6       # inner radius, m
mu, rho = .012, 1000. # viscosity, Pa s; density, kg/m³
sigma = .060          # surface tension, N/m
angle = 30.           # static advancing angle, degrees
g = 9.81              # m/s²
assert min(radius,mu,rho,sigma,g)>0 and 0 <= angle < 90

## Recover time from the meniscus-force balance
Use the fraction of equilibrium height as the independent coordinate. The formula for time is monotone and avoids starting an ODE solver at the formal infinite initial speed of this viscous approximation. The series at very small fractions avoids subtracting nearly equal numbers.

In [ ]:
def scales(a):
    head = 2*sigma*np.cos(np.deg2rad(angle))/(rho*g*a)
    mobility = rho*g*a*a/(8*mu)
    return head, head/mobility

def scaled_time(s):
    s = np.asarray(s,dtype=float)
    if np.any((s<0)|(s>=1)): raise ValueError("Require 0 ≤ height fraction < 1.")
    return np.where(s<1e-4,s*s*(.5+s/3+s*s/4),-s-np.log1p(-s))

H,tau = scales(radius)
for frac in (.25,.5,.9):
    print(f"{100*frac:.0f}% of {1000*H:.2f} mm at {tau*scaled_time(frac):.2f} s")
assert np.isclose(scaled_time(.5), np.log(2)-.5)
assert np.isclose(scaled_time(.00001)/(.00001**2/2),1.,rtol=1e-5)

## Reconstruct height as a function of laboratory time
When time is given, monotone bisection finds the height fraction. Each iteration halves the remaining interval independently for every entry in the requested time vector.

In [ ]:
def height_at(times, a):
    h, t = scales(a)
    times = np.asarray(times)
    lo, hi = np.zeros_like(times,dtype=float), np.ones_like(times,dtype=float)
    for _ in range(50):
        mid = (lo+hi)/2
        smaller = scaled_time(mid) < times/t
        lo,hi = np.where(smaller,mid,lo), np.where(smaller,hi,mid)
    return h*(lo+hi)/2

assert np.isclose(height_at(np.array([tau*scaled_time(.5)]),radius)[0], H/2)
sec = np.linspace(0,tau*2.5,400)
fig, ax = plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
for a,style in [(radius,"-"),(1.5*radius,"--")]:
    h,t = scales(a)
    ax[0].plot(sec,1000*height_at(sec,a),style,label=f"Radius {a*1e6:.0f} µm")
    ax[0].axhline(h*1000,ls=":",lw=.8,color="0.6")
f = np.linspace(.002,.93,240)
ax[1].plot(f,scaled_time(f)/(f*f/2),label="Gravity / early-law time")
ax[1].axhline(1,ls=":",color="0.5")
ax[0].set(xlabel="Elapsed time (s)",ylabel="Meniscus height (mm)",title="Two wetting tubes")
ax[1].set(xlabel="Fraction of static rise",ylabel="Time ratio",title="When the early approximation fails")
for p in ax:
    p.grid(alpha=.22); p.legend(fontsize=8)
plt.show()

## Infer an unknown and challenge the model
Choose a radius, record the time to 50% height, and hide the viscosity. Recover it from that single time using the known static head. A contaminated surface may change the contact angle. Decide which second observation would let you separate a change of viscosity from a change of capillary pressure.

## Set the camera clock and invert the time law
The film gives elapsed time and the theory gives time as a function of height. Invert that map for an entire list of frames using OneSimulation's vectorised bracketed inverse when it is installed. The browser uses the previously derived bisection. Compare the recovered heights to the dimensional marks.

In [ ]:
camera_times = tau * np.array([0.1, 0.5, 1.0])
try:
    from onesim.numerics.nonlinear.kernels.local_solvers.inversion.bracketed_inverse import bisect_increasing
except ImportError:
    checked_heights = height_at(camera_times, radius)
    method = "Notebook bisection"
else:
    target_times = camera_times / tau
    fractions = bisect_increasing(scaled_time, target_times, 0.0, 0.999)
    checked_heights = H * fractions
    method = "OneSimulation bracketed inverse"

assert np.allclose(tau * scaled_time(checked_heights/H), camera_times)
print(f"{method}: camera heights (mm)", np.round(1000 * checked_heights, 3))